# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/02017711723iot-dotcom/Flyrank_Internship_1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages are longer and younger on average than declining pages: about 3.2K vs 2.3K words and 184 vs 230 days of age.

**Methodology question:** The growth/decline label comes from `trend_direction`, which is calculated from the 30-day impression change versus the previous 30 days. This supports a descriptive comparison of growing and declining pages, but it does not establish that word count or age causes growth.

The evidence is useful as a directional signal, but the paper itself describes the comparison as observational. I would therefore use age and depth as decision-support features rather than claiming that increasing word count or reducing age will directly cause better performance.

### Finding 2 — The Content Performance Curve

The paper reports that content health peaks around 61–90 days and declines substantially around 271–365 days. It also reports a recovery among 365+ day content that was refreshed, while warning that the older refreshed result should be read narrowly.

**Methodology question:** The finding is based on comparisons across content-age and freshness groups using the portfolio data. The validation design supports an observed relationship between these groups, but it does not prove that content age itself causes the decline or that refreshing a page will always produce the reported improvement.

This is especially important for the 365+ result because the paper notes that the older active-content sample can introduce survivor bias. I would therefore describe age and freshness as observed/directional signals and use them to prioritize review rather than as guaranteed optimization levers.

**Overall methodology lesson:** The paper's strongest evidence comes from direct aggregate comparisons. Its ML appendix is explicitly exploratory, and the paper repeatedly distinguishes observed relationships from causal claims. I will use the same standard when evaluating my own model.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — basic paper-review check
print("Reviewed two FlyRank paper findings and identified the methodology questions.")

Reviewed two FlyRank paper findings and identified the methodology questions.


## 2. My model under an honest split (before/after)

### Before — Original row-level validation

In the earlier W05 setup, the model was evaluated using a row-level/stratified split. The Decision Tree achieved a Precision@50 of **0.680**.

This result is useful as an earlier reference, but pages from the same client could appear in both the training and test sets. This can make the estimate less representative of performance on completely unseen clients.

### After — Client-holdout validation

I re-ran the model using a client-holdout split so that no client appears in both training and testing data.

- Training rows: **23,837**
- Test rows: **6,163**
- Training clients: **25**
- Test clients: **7**
- Client overlap: **0**
- Training declining rate: **0.55**
- Test declining rate: **0.511**
- Decision Tree Precision@50: **0.620**

The Precision@50 decreased from **0.680** under the earlier row-level split to **0.620** under the client-holdout split.

This suggests that the earlier result was optimistic relative to the stricter unseen-client evaluation. I will use **0.620 Precision@50** as the more honest estimate for this model.

The model is intended to rank pages for human review, so this result should be treated as **decision-support evidence**, not as proof that the model will generalize to every future client.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compare the earlier row-level result with the honest client-holdout result.

before_p50 = 0.680
after_p50 = 0.620

print("Before - Row-level/stratified Precision@50:", before_p50)
print("After  - Client-holdout Precision@50:", after_p50)
print("Change in Precision@50:", round(after_p50 - before_p50, 3))

Before - Row-level/stratified Precision@50: 0.68
After  - Client-holdout Precision@50: 0.62
Change in Precision@50: -0.06


## 3. Leakage audit

The target is `is_declining_label`, which is created from `trend_direction`.

I checked the final model features against the label construction and removed label-derived fields from the feature set.

The final six features are:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`
- `word_count`

Potential leakage fields include:

- `trend_direction` — directly used to create the target.
- `trend_pct` — derived from the same trend comparison used for the target.

Neither field is included in the final model features.

The remaining features describe page state or historical performance and are not direct copies of the target. I therefore found no direct label leakage in the final six-feature model.

One limitation is that a feature can still create temporal leakage if its measurement window contains information that would not have been available at prediction time. For the current W05 model, I therefore treat the audit as a check for obvious direct leakage, not proof that every feature is temporally perfect.

This keeps the model suitable for ranking pages for human review rather than making automatic refresh decisions.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final model features
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Fields that should not be used because they are label/trend-derived
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

overlap = set(features).intersection(leakage_fields)

print("Final features:", features)
print("Potential leakage fields:", leakage_fields)
print("Feature/leakage overlap:", overlap)

if len(overlap) == 0:
    print("PASS: No direct label-derived fields are included in the final features.")
else:
    print("CHECK REQUIRED: Leakage field found in final features.")

Final features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Potential leakage fields: ['trend_direction', 'trend_pct', 'is_declining_label']
Feature/leakage overlap: set()
PASS: No direct label-derived fields are included in the final features.


## 4. Claim rewrite

### Claim 1 — Model performance

**Too strong:**  
"My model accurately predicts which pages will decline."

**Evidence-safe rewrite:**  
"On the client-holdout test set, the Decision Tree achieved a measured Precision@50 of 0.620 when ranking pages for review."

This is safer because the evaluation measures Precision@50 on one held-out set. It does not prove that the model will accurately predict every future page or client.

### Claim 2 — Feature importance

**Too strong:**  
"Impressions are the most important factor causing content decline."

**Evidence-safe rewrite:**  
"`impressions_90d` had the highest measured feature importance in the fitted Decision Tree, followed by `content_age_days` and `avg_position`."

Feature importance shows how the fitted model used the features. It does not establish that impressions cause content decline.

### Claim 3 — Recommended use

**Too strong:**  
"The model automatically identifies pages that should be refreshed."

**Evidence-safe rewrite:**  
"The model can provide a ranked decision-support queue for human review of potentially declining content."

This matches the actual purpose of the model and avoids claiming an automatic refresh decision.

### Overall claim

My model provides **directional, measured evidence** for prioritizing content review. The strongest result is the client-holdout Precision@50 of 0.620. I will avoid causal claims and avoid presenting the model as a guaranteed predictor or automatic decision-maker.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 records evidence-safe rewrites of the model's claims.

claims = {
    "performance": "Measured Precision@50 = 0.620 on the client-holdout test set.",
    "feature_importance": "impressions_90d had the highest measured feature importance.",
    "use": "Ranked decision-support queue for human review."
}

for name, claim in claims.items():
    print(f"{name}: {claim}")

performance: Measured Precision@50 = 0.620 on the client-holdout test set.
feature_importance: impressions_90d had the highest measured feature importance.
use: Ranked decision-support queue for human review.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.